# 🏁 34 · Capstone: an end-to-end visual-proteomics study and a paper audit
### One sealed sample, every tool of the course, and a reviewer's eye for your own results

**BiologyToResearch** · Part G — Frontiers, biology, research practice · Module 34 of 34

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/cryoem/tutorial/BiologyToResearch/notebooks/34_capstone_visual_proteomics.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 3–4 min for the baseline pipeline); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy, scikit-learn, Matplotlib |
| **Prerequisites** | All earlier modules; especially 07–14, 20 and 25 |
| **Estimated time** | 6–10 h as a project (≈ 90 min to run and read the baseline) |
| **Lab papers covered** | **AITom** (Zeng & Xu 2019; Zhan *et al.* 2025); **TomoMiner** (Frazier, Xu & Alber 2017); **Kim *et al.* 2023** (J. Mol. Biol. review); **DISCA** (Zeng *et al.* 2023) as the audited paper; and, through the pipeline, most methods of the course — see [`PAPER_ATLAS.md`](../PAPER_ATLAS.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis. The
> "sample" is an in silico cell built from real PDB models (downloaded at run time and cached outside the repository;
> offline, code-built stand-ins are used). Its composition is **sealed**: please do not decode it until Section 11 —
> the point of a capstone is to analyse data whose answer you do not know, and to evaluate only at the end.

## 🎯 Learning objectives

By the end of this capstone you will be able to:

1. **Run** a complete visual-proteomics pipeline on a simulated tomogram: picking → subtomograms → template-free
   classification → wedge-aware alignment and averaging → gold-standard FSC → identification → spatial statistics.
2. **Decide** the number of structural classes with the Bayesian information criterion, and **look inside** clusters
   for hidden heterogeneity.
3. **Recognise** when a class matches nothing in your reference library — a candidate **discovery** — using map
   reproducibility as the yardstick.
4. **Evaluate** every step against hidden ground truth only *after* the analysis, and **write** a structured report.
5. **Audit** a published paper with a structured worksheet (question, data, split unit, method, key equation, baseline,
   metric, failure case, falsification test).
6. **Plan** the move from toys to real data with AITom, EMPIAR, EMDB and the CZ cryoET Data Portal, following good
   research practice.

## 🗺️ Contents

| Section | Step of the study | Modules it uses |
|---|---|---|
| 0 · Setup | imports, toolkit | 01 |
| 1 · The sealed sample | simulate the tomogram | 06–08 |
| 2 · Picking | difference of Gaussians + elbow | 10 |
| 3 · Subtomograms and invariant features | extraction, recentring, features | 02, 03, 14 |
| 4 · Template-free classification | PCA + GMM, K by BIC | 04, 14, 20 |
| 5 · Alignment and averaging | wedge-aware search, wedge-weighted average | 07, 11–13 |
| 6 · Looking inside clusters | imputation + PCA + BIC | 13, 19 |
| 7 · Gold-standard resolution | independent half refinements, FSC | 09 |
| 8 · Identification and discovery | library matching vs reproducibility | 10, 14, 17 |
| 9 · Spatial statistics | concentrations, segregation test | 25 |
| 10 · The report | figures + table | — |
| 11 · Unsealing: evaluation | F1, ARI, identity, map vs truth | 04, 09 |
| 12 · Paper audit | worksheet, filled in for DISCA | all |
| 13 · From toys to real data | AITom, data portals, responsible research | all |

## 🧫 The biological question

You are given one tomogram of a cell you know little about. **Which macromolecular complexes does it contain, how many
copies of each, what do they look like, and where are they?** Answering this without assuming the answer is the dream
of *visual proteomics* — the thread that runs from the Xu lab's earliest papers (template matching and fitting with
Alber, Beck and Topf) through template-free pattern mining (MPP, DISCA) to today's foundation models. In this capstone
you play every role: microscopist (simulation), image analyst (pipeline), biologist (interpretation) and reviewer
(evaluation and audit).

---
## 0 · Environment setup

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — fine, everything runs on a CPU.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("matplotlib", "matplotlib"), ("sklearn", "scikit-learn")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import base64
import collections
import json
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import scipy.fft as sfft
from scipy import ndimage, stats
from scipy.spatial import cKDTree
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score
from sklearn.mixture import GaussianMixture

SEED = 26
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "image.cmap": "gray"})
T_START = time.time()
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v3.0; copied so this notebook runs alone) ──
import os
import struct

import numpy as np
from scipy import ndimage

ZOO = {
    "4v4r": dict(name="70S ribosome", organism="bacterial (E. coli / T. thermophilus)", mass_kDa=2330, symmetry="C1"),
    "4v7r": dict(name="80S ribosome", organism="S. cerevisiae", mass_kDa=3330, symmetry="C1"),
    "5mrc": dict(name="mitochondrial ribosome", organism="S. cerevisiae", mass_kDa=3320, symmetry="C1"),
    "3j9i": dict(name="20S proteasome", organism="Thermoplasma acidophilum", mass_kDa=710, symmetry="D7"),
    "6utj": dict(name="singly capped proteasome", organism="Thermoplasma acidophilum", mass_kDa=910, symmetry="C7"),
    "4cr2": dict(name="26S proteasome", organism="S. cerevisiae", mass_kDa=1280, symmetry="C1"),
    "4v94": dict(name="TRiC/CCT chaperonin (×2)", organism="S. cerevisiae", mass_kDa=1910, symmetry="C2"),
    "1bxn": dict(name="RuBisCO (L4S4 half)", organism="Cupriavidus necator", mass_kDa=290, symmetry="C4"),
    "1qvr": dict(name="ClpB (3 of 6 subunits)", organism="Thermus thermophilus", mass_kDa=290, symmetry="C1"),
    "3cf3": dict(name="p97/VCP ATPase", organism="Mus musculus", mass_kDa=570, symmetry="C6"),
    "2cg9": dict(name="Hsp90–Sba1 chaperone", organism="S. cerevisiae", mass_kDa=190, symmetry="C2"),
    "1u6g": dict(name="Cand1–Cul1–Roc1 ligase", organism="H. sapiens", mass_kDa=260, symmetry="C1"),
    "3d2f": dict(name="Sse1p–Hsp70 chaperone", organism="S. cerevisiae / H. sapiens", mass_kDa=230, symmetry="C1"),
    "1s3x": dict(name="Hsp70 ATPase domain", organism="H. sapiens", mass_kDa=44, symmetry="C1"),
}
_EXTRA_ENTRIES = {"5a1a": dict(name="β-galactosidase", organism="E. coli", mass_kDa=465, symmetry="D2")}
_BOND = dict(N_CA=1.458, CA_C=1.525, C_N=1.329, C_O=1.231)
_ANGLE = dict(N_CA_C=111.2, CA_C_N=116.2, C_N_CA=121.7, CA_C_O=120.5)
_SIDECHAIN = {
    "GLY": [], "ALA": [], "SER": [("OG", "O")], "CYS": [("SG", "S")], "VAL": [("CG1", "C"), ("CG2", "C")],
    "THR": [("OG1", "O"), ("CG2", "C")], "PRO": [("CG", "C"), ("CD", "C")],
    "LEU": [("CG", "C"), ("CD1", "C"), ("CD2", "C")], "ILE": [("CG1", "C"), ("CG2", "C"), ("CD1", "C")],
    "ASP": [("CG", "C"), ("OD1", "O"), ("OD2", "O")], "ASN": [("CG", "C"), ("OD1", "O"), ("ND2", "N")],
    "GLU": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("OE2", "O")], "GLN": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("NE2", "N")],
    "LYS": [("CG", "C"), ("CD", "C"), ("CE", "C"), ("NZ", "N")], "MET": [("CG", "C"), ("SD", "S"), ("CE", "C")],
    "ARG": [("CG", "C"), ("CD", "C"), ("NE", "N"), ("CZ", "C"), ("NH1", "N"), ("NH2", "N")],
    "HIS": [("CG", "C"), ("ND1", "N"), ("CD2", "C"), ("CE1", "C"), ("NE2", "N")],
    "PHE": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C")],
    "TYR": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C"), ("OH", "O")],
    "TRP": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("NE1", "N"), ("CE2", "C"), ("CE3", "C"), ("CZ2", "C"),
            ("CZ3", "C"), ("CH2", "C")],
}
_AA_FREQ = {"LEU": 9.9, "ALA": 8.3, "GLY": 7.1, "VAL": 6.9, "GLU": 6.8, "SER": 6.6, "ILE": 5.9, "LYS": 5.8, "ARG": 5.5,
            "ASP": 5.5, "THR": 5.3, "PRO": 4.7, "ASN": 4.1, "GLN": 3.9, "PHE": 3.9, "TYR": 2.9, "MET": 2.4, "HIS": 2.3,
            "CYS": 1.4, "TRP": 1.1}
def _nerf(a, b, c, bond, angle, torsion):
    """Place atom d so that |cd| = bond, angle(b,c,d) = angle and dihedral(a,b,c,d) = torsion (degrees)."""
    bc = (c - b) / np.linalg.norm(c - b)
    n = np.cross(b - a, bc)
    n /= np.linalg.norm(n)
    m = np.cross(n, bc)
    ang, tor = np.radians(angle), np.radians(torsion)
    d = np.array([-bond * np.cos(ang), bond * np.sin(ang) * np.cos(tor), bond * np.sin(ang) * np.sin(tor)])
    return c + d[0] * bc + d[1] * m + d[2] * n


def build_backbone(phi, psi, omega=None) -> np.ndarray:
    """Backbone atoms N, CA, C, O, CB (array n×5×3, Å) of a chain with the given torsion angles (degrees)."""
    n = len(phi)
    omega = np.full(n, 180.0) if omega is None else np.asarray(omega, float)
    N = [np.zeros(3)]
    CA = [np.array([_BOND["N_CA"], 0.0, 0.0])]
    C = [_nerf(np.array([0.0, 1.0, 0.0]), N[0], CA[0], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[0])]
    for i in range(n - 1):
        N.append(_nerf(N[i], CA[i], C[i], _BOND["C_N"], _ANGLE["CA_C_N"], psi[i]))
        CA.append(_nerf(CA[i], C[i], N[i + 1], _BOND["N_CA"], _ANGLE["C_N_CA"], omega[i]))
        C.append(_nerf(C[i], N[i + 1], CA[i + 1], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[i + 1]))
    N, CA, C = np.array(N), np.array(CA), np.array(C)
    O = np.array([_nerf(N[i], CA[i], C[i], _BOND["C_O"], _ANGLE["CA_C_O"], psi[i] + 180.0) for i in range(n)])
    b, c = CA - N, C - CA
    a = np.cross(b, c)
    CB = -0.58273431 * a + 0.56802827 * b - 0.54067466 * c + CA      # ideal Cβ position
    return np.stack([N, CA, C, O, CB], 1)


def _ideal_helix_template(n=160):
    """An ideal α-helix (φ = −57.8°, ψ = −47.0°) with its axis along +z through x = y = 0."""
    bb = build_backbone(np.full(n, -57.8), np.full(n, -47.0))
    ca = bb[:, 1]
    centre = ca.mean(0)
    u = np.linalg.svd(ca - centre)[2][0]
    if np.dot(ca[-1] - ca[0], u) < 0:
        u = -u
    # rotation taking u to z
    v = np.cross(u, [0, 0, 1.0]); s = np.linalg.norm(v); cth = u[2]
    K = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
    Rm = np.eye(3) + K + K @ K * ((1 - cth) / (s ** 2 + 1e-12))
    xyz = (bb - centre) @ Rm.T
    axis_xy = xyz[:, 1, :2].mean(0)
    xyz[..., :2] -= axis_xy
    return xyz


_HELIX = _ideal_helix_template()
def _sidechains(bb, resnames, rng):
    """Approximate side-chain heavy atoms grown outward from Cβ (positions are schematic; counts/elements are right)."""
    names, elems, xyz, resid = [], [], [], []
    ca, cb = bb[:, 1], bb[:, 4]
    out = cb - ca
    out /= np.linalg.norm(out, axis=1, keepdims=True)
    for i, rn in enumerate(resnames):
        sc = _SIDECHAIN[rn]
        if not sc:
            continue
        perp = np.cross(out[i], rng.normal(size=3)); perp /= np.linalg.norm(perp)
        for k, (an, el) in enumerate(sc):
            depth = 1.0 + (k // 2)                               # schematic zig-zag outward
            pos = cb[i] + 1.5 * depth * out[i] + 0.9 * (1 if k % 2 else -1) * perp
            names.append(an); elems.append(el); xyz.append(pos); resid.append(i)
    return names, elems, np.array(xyz).reshape(-1, 3), np.array(resid, int)


def _loop_path(a, b, outward, step=3.8):
    """Cα positions of a loop from Cα `a` to Cα `b`: a circular arc bulging along `outward`, cut into equal chords so
    that every Cα–Cα distance (including the two ends) is exactly `step` Å."""
    c = np.linalg.norm(b - a)
    e = (b - a) / c
    u = outward - (outward @ e) * e                                  # bulge direction ⟂ to the chord
    if np.linalg.norm(u) < 1e-6:
        u = np.cross(e, [0.0, 0.0, 1.0]) if abs(e[2]) < 0.9 else np.cross(e, [1.0, 0.0, 0.0])
    u = u / np.linalg.norm(u)
    m = max(int(np.ceil(c / step)), 1)                               # m new residues → n = m + 1 equal chords
    n = m + 1
    lo, hi = 1e-9, 2 * np.pi - 1e-9                                  # arc angle Θ: sin(Θ/2) / sin(Θ/2n) = c / step
    for _ in range(60):
        th = (lo + hi) / 2
        lo, hi = (th, hi) if np.sin(th / 2) / np.sin(th / (2 * n)) > c / step else (lo, th)
    R = step / (2 * np.sin(th / (2 * n)))
    centre = (a + b) / 2 - R * np.cos(th / 2) * u
    beta = np.pi / 2 + th / 2 - th * np.arange(1, n) / n             # angles of the new points, a → b over the bulge
    return centre + R * (np.outer(np.cos(beta), e) + np.outer(np.sin(beta), u))


def _assemble_chain(ca_path_segments, rng, chain="A", start_resnum=1):
    """Turn helices (n×5×3 arrays) joined by loops into one chain with residue names and side chains."""
    blocks = []
    for k, seg in enumerate(ca_path_segments):
        blocks.append(seg)
        if k + 1 < len(ca_path_segments):
            a, b = seg[-1, 1], ca_path_segments[k + 1][0, 1]
            path = _loop_path(a, b, (a + b) / 2 - seg.reshape(-1, 3).mean(0))
            m = len(path)
            tang = np.gradient(path, axis=0) if m > 1 else np.array([b - a])
            tang /= np.linalg.norm(tang, axis=1, keepdims=True)
            side = np.cross(tang, rng.normal(size=3)); side /= np.linalg.norm(side, axis=1, keepdims=True)
            loop = np.stack([path - 1.2 * tang, path, path + 1.2 * tang, path + 1.2 * tang + 1.0 * side,
                             path - 1.3 * side], 1)
            blocks.append(loop)
    bb = np.concatenate(blocks, 0)
    aa = list(_AA_FREQ)
    p = np.array([_AA_FREQ[a] for a in aa]); p /= p.sum()
    resnames = rng.choice(aa, len(bb), p=p)
    bb_names = ["N", "CA", "C", "O", "CB"]
    names, elems, xyz, resid = [], [], [], []
    for i in range(len(bb)):
        for j, an in enumerate(bb_names):
            if an == "CB" and resnames[i] == "GLY":
                continue
            names.append(an); elems.append(an[0]); xyz.append(bb[i, j]); resid.append(i)
    sn, se, sx, sr = _sidechains(bb, resnames, rng)
    names += sn; elems += se; resid = np.concatenate([np.array(resid, int), sr])
    xyz = np.concatenate([np.array(xyz), sx], 0)
    order = np.argsort(resid, kind="stable")
    return dict(xyz=xyz[order].astype(np.float32), name=np.array(names)[order], element=np.array(elems)[order],
                resname=resnames[resid[order]], resnum=(resid[order] + start_resnum).astype(int),
                chain=np.full(len(order), chain))


def helix_bundle(radii_A, rng, chain="A", spacing=10.0):
    """A globular 'domain' that fills an ellipsoid (semi-axes in Å, long axis = local z) with antiparallel helices."""
    a, b, c = radii_A[0] + 2.5, radii_A[1] + 2.5, radii_A[2] + 1.5   # helices stop short of the surface: pad
    rows = []
    j = 0
    while (j * spacing * np.sqrt(3) / 2) <= b:
        rows.append(j); rows.append(-j) if j else None; j += 1
    pts = []
    for jj in sorted(set(rows)):
        y = jj * spacing * np.sqrt(3) / 2
        xs = np.arange(-a, a + 1e-6, spacing) + (spacing / 2 if jj % 2 else 0)
        row = [(x, y) for x in xs if (x / a) ** 2 + (y / b) ** 2 < 1]
        pts.append(row if (jj % 2 == 0) else row[::-1])
    pts = [p for row in pts for p in row]
    segs = []
    for k, (x, y) in enumerate(pts):
        h = c * np.sqrt(max(0.0, 1 - (x / a) ** 2 - (y / b) ** 2)) - 2.0
        n = int(2 * h / 1.5)
        if n < 6:
            continue
        seg = _HELIX[:n].copy()
        seg[..., 2] -= seg[:, 1, 2].mean()
        phase = rng.uniform(0, 2 * np.pi)
        Rz = np.array([[np.cos(phase), -np.sin(phase), 0], [np.sin(phase), np.cos(phase), 0], [0, 0, 1]])
        seg = seg @ Rz.T
        if len(segs) % 2 == 1:                                   # antiparallel neighbours: 180° turn about x
            seg = seg * np.array([1.0, -1.0, -1.0])
        seg = seg + np.array([x, y, rng.uniform(-1.0, 1.0)])
        segs.append(seg)
    return _assemble_chain(segs, rng, chain)


def _transform(atoms, R=np.eye(3), t=(0, 0, 0)):
    out = dict(atoms)
    out["xyz"] = (atoms["xyz"] @ np.asarray(R, float).T + np.asarray(t, float)).astype(np.float32)
    return out


def merge_atoms(parts):
    keys = parts[0].keys()
    return {k: np.concatenate([p[k] for p in parts]) for k in keys}


def _rz(a):
    return np.array([[np.cos(a), -np.sin(a), 0], [np.sin(a), np.cos(a), 0], [0, 0, 1.0]])


def _rx(a):
    return np.array([[1, 0, 0], [0, np.cos(a), -np.sin(a)], [0, np.sin(a), np.cos(a)]])


def _ry(a):
    return np.array([[np.cos(a), 0, np.sin(a)], [0, 1, 0], [-np.sin(a), 0, np.cos(a)]])


_CHAIN_IDS = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789"
def _ring(n, radius, z, radii, phase=0.0, tilt=None):
    """n identical subunits (semi-axes `radii` nm: radial, tangential, axial) around the z axis."""
    out = []
    for k in range(n):
        a = phase + 2 * np.pi * k / n
        R = _rz(a) if tilt is None else _rz(a) @ tilt
        out.append(dict(radii=radii, centre=(radius * np.cos(a), radius * np.sin(a), z), R=R, key=("ring", radii, tilt is None)))
    return out


def _lobe(radii, centre, R=None):
    return [dict(radii=radii, centre=centre, R=np.eye(3) if R is None else R, key=None)]


def _dihedral(subunits):
    """Add the partners generated by a 2-fold axis along x (dihedral symmetry)."""
    Rx = _rx(np.pi)
    return subunits + [dict(radii=s["radii"], centre=tuple(Rx @ np.array(s["centre"])), R=Rx @ s["R"], key=s["key"])
                       for s in subunits]


def _blueprint(pid):
    """Subunit layout (nm) of the code-generated stand-in for each zoo entry."""
    if pid in ("4v4r", "4v7r", "5mrc"):
        s = {"4v4r": 1.0, "4v7r": 1.13, "5mrc": 1.14}[pid]
        L = [_lobe((9.0 * s, 8.0 * s, 6.5 * s), (0, 0, 4.5 * s)),                       # large subunit body
             _lobe((3.0 * s, 3.0 * s, 3.0 * s), (0, 1.0 * s, 11.0 * s)),                # central protuberance
             _lobe((2.0 * s, 2.0 * s, 3.5 * s), (-9.0 * s, 0, 7.0 * s), _ry(-0.6)),     # L1-like stalk
             _lobe((1.4 * s, 1.4 * s, 4.0 * s), (9.5 * s, 0, 6.5 * s), _ry(0.7)),       # L7/L12-like stalk
             _lobe((7.5 * s, 5.0 * s, 3.8 * s), (1.0 * s, 0, -4.5 * s)),                # small subunit body
             _lobe((3.5 * s, 3.2 * s, 3.0 * s), (-5.5 * s, -1.0 * s, -6.5 * s))]        # small subunit head
        if pid == "4v7r":                                                                # eukaryotic expansion segments
            L += [_lobe((3.2, 2.6, 2.8), (8.0, 7.0, 7.0)), _lobe((3.0, 2.6, 3.2), (-6.0, -8.0, -6.5)),
                  _lobe((2.4, 2.4, 3.6), (-10.5, 4.0, -1.0), _ry(0.5))]
        if pid == "5mrc":                                                                # mitochondria-specific proteins
            L += [_lobe((4.2, 3.6, 2.8), (0, -7.5, 12.5)), _lobe((3.4, 3.0, 3.4), (9.0, -6.0, -8.0)),
                  _lobe((3.0, 3.0, 3.0), (-9.5, 7.0, 1.0)), _lobe((2.5, 4.5, 2.0), (2.0, 9.0, 9.0))]
        prng = np.random.default_rng(_seed(pid) + 1)                                     # rRNA-like surface bumps
        for _ in range(8):
            d = prng.normal(size=3); d /= np.linalg.norm(d)
            L += [_lobe(tuple(prng.uniform(1.2, 1.9, 3)), tuple(s * d * np.array([9.5, 8.0, 10.5])), _rz(prng.uniform(0, 6)))]
        return [x for l in L for x in l]
    if pid in ("3j9i", "6utj", "4cr2"):
        core = []
        for z, radii, ph in [(1.9, (1.9, 1.7, 2.0), 0.0), (5.6, (2.1, 1.7, 1.85), np.pi / 7)]:
            core += _ring(7, 3.6, z, radii, ph)
        core = _dihedral(core)                                                           # α7β7β7α7, D7
        if pid == "6utj":
            core += _ring(7, 3.3, 10.0, (1.9, 1.6, 2.4), np.pi / 14)                     # one heptameric cap: C7
        if pid == "4cr2":
            core += _ring(6, 4.3, 9.6, (2.2, 1.8, 2.0), 0.3)                             # base ring of six
            core += (_lobe((4.0, 3.0, 3.0), (3.5, 0, 14.5)) + _lobe((3.0, 2.5, 3.5), (-1.5, 3.0, 15.5))
                     + _lobe((2.5, 2.5, 2.5), (-3.5, -2.0, 13.5)) + _lobe((3.0, 3.0, 2.5), (-4.5, 0.5, 10.5)))
        return core
    if pid == "4v94":
        one = []
        for z in (3.6, -3.6):
            one += _ring(8, 5.2, z, (2.6, 2.0, 3.4), np.pi / 8 if z < 0 else 0.0)
        two = []
        for dx, rot in [(-8.5, 0.0), (8.5, np.pi)]:
            for s in one:
                c = _rz(rot) @ np.array(s["centre"])
                two.append(dict(radii=s["radii"], centre=(c[0] + dx, c[1], c[2]), R=_rz(rot) @ s["R"], key=s["key"]))
        return two
    if pid == "1bxn":
        # like the deposited file (the crystal's asymmetric unit): one L4S4 layer = half of the L8S8 enzyme, C4.
        # `_dihedral` of this list would give the complete enzyme (D4).
        return _ring(4, 3.6, 2.2, (2.6, 2.4, 2.4)) + _ring(4, 2.6, 5.0, (1.9, 1.7, 1.35), np.pi / 4)
    if pid == "1qvr":
        # like the deposited file: three neighbouring subunits (half of the C6 hexameric ring), C1
        tiers = [_ring(6, 4.2, 2.2, (2.2, 2.0, 2.0)), _ring(6, 3.9, -2.2, (2.3, 2.0, 2.0), np.pi / 6),
                 _ring(6, 7.2, 2.4, (0.9, 0.9, 4.6), 0.4, tilt=_rx(1.2)),               # coiled-coil 'propellers'
                 _ring(6, 3.8, 6.6, (1.7, 1.6, 1.8), 0.5)]                               # N-domains on top
        return [s for t in tiers for s in t[:3]]
    if pid == "3cf3":
        return (_ring(6, 4.0, 2.0, (2.1, 2.0, 2.0)) + _ring(6, 4.0, -2.2, (2.2, 2.0, 2.1), np.pi / 6)
                + _ring(6, 6.6, 3.0, (2.0, 1.6, 1.8), 0.25))                             # N-domains flush: C6
    if pid == "2cg9":
        half = (_lobe((1.9, 1.7, 1.7), (2.2, 0.8, 6.0)) + _lobe((2.2, 1.8, 2.0), (2.6, -0.4, 2.0))
                + _lobe((1.7, 1.5, 1.8), (1.4, 0.3, -2.2)) + _lobe((1.4, 1.2, 1.2), (4.2, 1.6, 7.6)))
        Rz = _rz(np.pi)
        return half + [dict(radii=h["radii"], centre=tuple(Rz @ np.array(h["centre"])), R=Rz @ h["R"], key=None) for h in half]
    if pid == "1u6g":
        out = []
        for k in range(6):                                                                # curved cullin stalk
            t = k / 5
            out += _lobe((1.4, 1.4, 1.6), (8.0 * np.sin(0.9 * t) - 3.0, 0, 9.0 * t - 4.5), _ry(0.9 * t))
        out += _lobe((2.2, 2.0, 2.0), (4.0, 0, 5.8)) + _lobe((1.2, 1.1, 1.1), (6.0, 1.8, 6.5))
        for k in range(10):                                                               # Cand1-like superhelix
            a = 1.6 * np.pi * k / 9
            out += _lobe((1.5, 1.4, 1.4), (-3.0 + 7.0 * np.sin(0.9 * k / 9) + 2.6 * np.cos(a), 2.6 * np.sin(a), 9.0 * k / 9 - 4.5))
        return out
    if pid == "3d2f":
        comp = (_lobe((1.8, 1.6, 1.8), (0, 0, 0)) + _lobe((1.8, 1.6, 1.8), (0, 3.2, 0)) + _lobe((1.6, 1.4, 2.4), (3.0, 1.6, 1.8))
                + _lobe((1.6, 1.5, 1.6), (-3.2, 1.0, -1.0)) + _lobe((1.6, 1.5, 1.6), (-3.2, 3.6, -0.6)))
        R2, t2 = _rx(2.3) @ _rz(0.7), np.array([1.5, -4.5, 5.5])
        return comp + [dict(radii=c["radii"], centre=tuple(R2 @ np.array(c["centre"]) + t2), R=R2 @ c["R"], key=None) for c in comp]
    if pid == "1s3x":
        return (_lobe((1.4, 1.4, 1.6), (-1.6, 0, -1.2)) + _lobe((1.3, 1.3, 1.5), (-1.4, 0.2, 1.6))
                + _lobe((1.4, 1.4, 1.6), (1.6, 0, -1.2)) + _lobe((1.2, 1.3, 1.4), (1.5, -0.3, 1.8)))
    raise KeyError(pid)


def _seed(pid):
    return int.from_bytes(pid.encode(), "little") % (2 ** 31)


_SCALE = {"4v4r": 0.923, "4v7r": 0.906, "5mrc": 0.883, "3j9i": 0.930, "6utj": 0.930, "4cr2": 0.930, "4v94": 0.886,
          "1bxn": 0.930, "1qvr": 0.885, "3cf3": 0.935, "2cg9": 0.995, "1u6g": 0.934, "3d2f": 0.997, "1s3x": 0.884}
def standin_atoms(pid):
    """Atomic model (dict of arrays) of the code-built stand-in for zoo entry `pid` (Å, centred at the origin).

    Used when the real entry cannot be downloaded: ideal α-helices arranged with the size, mass and symmetry of the
    PDB entry. Deterministic (the random numbers are seeded by the entry's name)."""
    pid = pid.lower()
    rng = np.random.default_rng(_seed(pid))
    built, parts = {}, []
    f = _SCALE.get(pid, 1.0)
    for k, s in enumerate(_blueprint(pid)):
        radii_A = tuple(10.0 * f * r for r in s["radii"])
        key = s["key"]
        if key is not None and key in built:
            sub = built[key]
        else:
            sub = helix_bundle(radii_A, rng)
            if key is not None:
                built[key] = sub
        part = _transform(sub, s["R"], 10.0 * f * np.array(s["centre"]))
        part["chain"] = np.full(len(part["xyz"]), _CHAIN_IDS[k % len(_CHAIN_IDS)])
        parts.append(part)
    atoms = merge_atoms(parts)
    if pid in ("3j9i", "6utj", "4cr2"):                             # lay the proteasome barrels along x (side view down z)
        atoms["xyz"] = (atoms["xyz"] @ _ry(np.pi / 2).T).astype(np.float32)
    atoms["xyz"] -= atoms["xyz"].mean(0)
    return atoms


def pdb_cache_dir():
    """Folder (outside the repository) where downloaded PDB files are kept."""
    return os.environ.get("BIO2RESEARCH_PDB_CACHE") or os.path.join(os.path.expanduser("~"), ".cache",
                                                                    "biology_to_research", "pdb")


def _pdb_urls(pid, kind):
    """Download addresses of one PDB file: kind = 'entry' or 'assembly1' (mmCIF) or 'pdb' (legacy PDB format)."""
    pid = pid.lower()
    mid = pid[1:3]
    if kind == "entry":
        return [f"https://files.rcsb.org/download/{pid.upper()}.cif.gz",
                f"https://files.wwpdb.org/pub/pdb/data/structures/divided/mmCIF/{mid}/{pid}.cif.gz",
                f"https://www.ebi.ac.uk/pdbe/entry-files/download/{pid}.cif"]
    if kind == "assembly1":
        return [f"https://files.rcsb.org/download/{pid.upper()}-assembly1.cif.gz",
                f"https://files.wwpdb.org/pub/pdb/data/assemblies/mmCIF/divided/{mid}/{pid}-assembly1.cif.gz"]
    if kind == "pdb":
        return [f"https://files.rcsb.org/download/{pid.upper()}.pdb.gz",
                f"https://files.wwpdb.org/pub/pdb/data/structures/divided/pdb/{mid}/pdb{pid}.ent.gz",
                f"https://www.ebi.ac.uk/pdbe/entry-files/download/pdb{pid}.ent"]
    raise ValueError(kind)


def fetch_pdb(pid, kind="entry", timeout=60):
    """Text of a PDB file, downloaded once and cached outside the repository. Returns (text, where_from), or
    (None, reason) when no mirror can be reached (or BIO2RESEARCH_STANDINS is set)."""
    import gzip
    import urllib.request
    name = {"entry": f"{pid.lower()}.cif.gz", "assembly1": f"{pid.lower()}-assembly1.cif.gz",
            "pdb": f"{pid.lower()}.pdb.gz"}[kind]
    path = os.path.join(pdb_cache_dir(), name)
    if os.path.exists(path):
        return gzip.decompress(open(path, "rb").read()).decode("utf-8", "replace"), f"cached copy ({path})"
    if os.environ.get("BIO2RESEARCH_STANDINS"):
        return None, "BIO2RESEARCH_STANDINS is set"
    reason = "no mirror tried"
    for url in _pdb_urls(pid, kind):
        try:
            with urllib.request.urlopen(url, timeout=timeout) as response:
                raw = response.read()
            text = (gzip.decompress(raw) if raw[:2] == b"\x1f\x8b" else raw).decode("utf-8", "replace")
            if ("_atom_site." not in text) and ("\nATOM  " not in text):
                raise ValueError("no atoms in the downloaded file")
            os.makedirs(pdb_cache_dir(), exist_ok=True)
            with open(path, "wb") as f:
                f.write(gzip.compress(text.encode()))
            return text, url
        except Exception as e:  # noqa: BLE001 — any failure: try the next mirror
            reason = f"{type(e).__name__}: {str(e)[:120]}"
    return None, reason


def _cif_tokens(line):
    """Split one line of an mmCIF loop into values (quoted values may contain spaces)."""
    if "'" not in line and '"' not in line:
        return line.split()
    import re
    toks = re.findall(r"'(?:[^']|'(?=\S))*'|\"(?:[^\"]|\"(?=\S))*\"|\S+", line)
    return [t[1:-1] if len(t) > 1 and t[0] == t[-1] and t[0] in "'\"" else t for t in toks]


_WATER = {"HOH", "DOD", "WAT", "H2O"}
def read_mmcif_atoms(text):
    """Atoms of the first model in an mmCIF file → dict(xyz, name, element, resname, chain, resnum).

    Reads the `_atom_site` loop; skips water and hydrogen, and keeps only the first of alternative atom locations."""
    lines = text.splitlines()
    i = next(k for k, line in enumerate(lines) if line.startswith("_atom_site."))
    cols = []
    while i < len(lines) and lines[i].startswith("_atom_site."):
        cols.append(lines[i].split()[0][len("_atom_site."):])
        i += 1
    c = {name: k for k, name in enumerate(cols)}
    get = lambda *names: next((c[n] for n in names if n in c), None)   # noqa: E731
    ix, iy, iz = c["Cartn_x"], c["Cartn_y"], c["Cartn_z"]
    i_el, i_name = get("type_symbol"), get("auth_atom_id", "label_atom_id")
    i_res, i_ch = get("auth_comp_id", "label_comp_id"), get("auth_asym_id", "label_asym_id")
    i_num, i_alt, i_mod = get("auth_seq_id", "label_seq_id"), get("label_alt_id"), get("pdbx_PDB_model_num")
    xyz, name, element, resname, chain, resnum = [], [], [], [], [], []
    first_model, buf = None, []
    for line in lines[i:]:
        if not line or line.startswith(("#", "_", "loop_", "data_")):
            break
        buf += _cif_tokens(line)
        while len(buf) >= len(cols):
            row, buf = buf[:len(cols)], buf[len(cols):]
            if i_mod is not None:
                first_model = row[i_mod] if first_model is None else first_model
                if row[i_mod] != first_model:
                    continue
            el = row[i_el].upper() if i_el is not None else row[i_name][:1].upper()
            if el in ("H", "D") or row[i_res] in _WATER:
                continue
            if i_alt is not None and row[i_alt] not in (".", "?", "A", "1"):
                continue
            xyz.append((float(row[ix]), float(row[iy]), float(row[iz])))
            name.append(row[i_name])
            element.append(el.capitalize() if len(el) > 1 else el)
            resname.append(row[i_res])
            chain.append(row[i_ch])
            num = row[i_num] if i_num is not None else "0"
            resnum.append(int(num) if num.lstrip("-").isdigit() else 0)
    return dict(xyz=np.array(xyz, np.float32), name=np.array(name), element=np.array(element),
                resname=np.array(resname), chain=np.array(chain), resnum=np.array(resnum))


_ATOMIC_MASS = {"H": 1.008, "C": 12.011, "N": 14.007, "O": 15.999, "P": 30.974, "S": 32.06, "Se": 78.97, "Mg": 24.305,
                "Zn": 65.38, "Fe": 55.845, "Mn": 54.938, "K": 39.098, "Na": 22.99, "Cl": 35.45, "Ca": 40.078}
def model_mass_kDa(atoms):
    """Mass of an atomic model without hydrogens, plus ~8 % for the missing hydrogens (kDa)."""
    return 1.08 * sum(_ATOMIC_MASS.get(e, 12.0) for e in atoms["element"]) / 1000.0


def canonical_frame(atoms):
    """Centre a model and rotate it onto its principal axes — longest extent along x, shortest along z — with the
    signs fixed by the skewness of the atom distribution. A proteasome barrel then lies along x and a ring
    (RuBisCO, p97) lies in the x–y plane, like the stand-ins."""
    xyz = atoms["xyz"].astype(np.float64)
    X = xyz - xyz.mean(0)
    evecs = np.linalg.eigh(np.cov(X.T))[1][:, ::-1]                 # columns: principal axes, largest first
    Y = X @ evecs
    sign = np.sign(np.sum(Y ** 3, axis=0))
    sign[sign == 0] = 1.0
    Y *= sign
    if np.linalg.det(evecs * sign) < 0:                             # keep a proper rotation (no mirror image)
        Y[:, 2] *= -1
    out = dict(atoms)
    out["xyz"] = Y.astype(np.float32)
    return out


def pdb_atoms(pid, verbose=False):
    """Real atomic model of a PDB entry, in the canonical frame. Returns (atoms, description), or (None, reason).

    Uses the deposited entry unless its mass is far from ZOO's reference mass, in which case the first biological
    assembly is tried and the closer of the two kept (e.g. one ribosome rather than the two of a crystal)."""
    pid = pid.lower()
    ref = ZOO.get(pid, {}).get("mass_kDa") or _EXTRA_ENTRIES.get(pid, {}).get("mass_kDa")
    candidates, reason = [], ""
    for kind in ("entry", "assembly1"):
        if candidates and (ref is None or abs(np.log(model_mass_kDa(candidates[0][0]) / ref)) < np.log(1.3)):
            break
        text, where = fetch_pdb(pid, kind)
        if text is None:
            reason = where
            continue
        try:
            atoms = read_mmcif_atoms(text)
        except (StopIteration, KeyError, ValueError) as e:
            reason = f"could not read the file ({type(e).__name__})"
            continue
        if len(atoms["xyz"]):
            label = "deposited entry" if kind == "entry" else "biological assembly 1"
            candidates.append((atoms, f"PDB {pid.upper()} ({label}; {len(atoms['xyz']):,} atoms; from {where})"))
    if not candidates:
        return None, reason
    atoms, desc = min(candidates, key=lambda a: abs(np.log(model_mass_kDa(a[0]) / ref)) if ref else 0.0)
    if verbose:
        print(desc)
    return canonical_frame(atoms), desc


_ATOMS_CACHE = {}
def zoo_atoms(pid):
    """Atomic model (dict of arrays, Å, centred) of zoo entry `pid`: the real PDB model when it can be downloaded, the
    code-built stand-in otherwise. The source is recorded in `atoms_source(pid)`."""
    pid = pid.lower()
    if pid not in _ATOMS_CACHE:
        atoms, desc = pdb_atoms(pid)
        if atoms is None:
            atoms, desc = standin_atoms(pid), f"code-built stand-in modelled on PDB {pid.upper()} (download failed: {desc})"
        _ATOMS_CACHE[pid] = (atoms, desc)
    return _ATOMS_CACHE[pid][0]


def atoms_source(pid):
    """Where the atoms of zoo entry `pid` came from (a real PDB file or the offline stand-in)."""
    zoo_atoms(pid)
    return _ATOMS_CACHE[pid.lower()][1]


_Z = {"C": 6.0, "N": 7.0, "O": 8.0, "S": 16.0, "P": 15.0, "H": 1.0, "Mg": 12.0, "Zn": 30.0, "Fe": 26.0, "K": 19.0,
      "Na": 11.0, "Cl": 17.0, "Ca": 20.0, "Mn": 25.0, "Se": 34.0}
def atoms_to_map(xyz, weights=None, voxel=10.0, box=None, sigma_A=None):
    """Density map [z, y, x] from atom positions (Å): trilinear splatting on a grid twice as fine, Gaussian blur
    (σ = `sigma_A`, default = voxel/2), then 2×2×2 binning to `voxel` Å. Centred on the box if `box` is given."""
    xyz = np.asarray(xyz, float)
    w = np.ones(len(xyz)) if weights is None else np.asarray(weights, float)
    fine = voxel / 2.0
    if box is None:
        ext = np.ptp(xyz, 0).max() + 6 * voxel
        box = int(np.ceil(ext / voxel)) + 1
    n = 2 * box
    centre = (n - 1) / 2.0
    p = xyz[:, ::-1] / fine + centre                                  # (z, y, x) grid coordinates
    grid = np.zeros((n, n, n), np.float32)
    i0 = np.floor(p).astype(int)
    f = p - i0
    for dz in (0, 1):
        for dy in (0, 1):
            for dx in (0, 1):
                idx = i0 + np.array([dz, dy, dx])
                wt = w * np.where(dz, f[:, 0], 1 - f[:, 0]) * np.where(dy, f[:, 1], 1 - f[:, 1]) * np.where(dx, f[:, 2], 1 - f[:, 2])
                ok = np.all((idx >= 0) & (idx < n), 1)
                np.add.at(grid, tuple(idx[ok].T), wt[ok])
    grid = ndimage.gaussian_filter(grid, (sigma_A if sigma_A else voxel / 2.0) / fine)
    return grid.reshape(box, 2, box, 2, box, 2).mean((1, 3, 5)).astype(np.float32)


def zoo_map(pid, voxel="10A"):
    """(density[z, y, x], voxel size Å, source) of a zoo entry, painted from its atoms (`zoo_atoms`)."""
    atoms = zoo_atoms(pid)
    v = float(str(voxel).rstrip("A"))
    w = np.array([_Z.get(e, 6.0) for e in atoms["element"]])
    vol = atoms_to_map(atoms["xyz"], w, voxel=v)
    return vol, v, atoms_source(pid)


def prepare_map(vol: np.ndarray, size: int = 32, sigma: float = 1.0) -> np.ndarray:
    """Low-pass, centre (by centre of mass), crop/pad to ``size``³ and scale to [0, 1].

    Zoo maps are sharp sums of atoms; a Gaussian blur of ~1 voxel (1 nm)
    mimics the resolution of a cellular tomogram.
    """
    v = ndimage.gaussian_filter(np.clip(vol, 0, None).astype(np.float32), sigma)
    s = np.array(v.shape)
    src0 = np.maximum((s - size) // 2, 0)                  # where the crop starts in the input ...
    dst0 = np.maximum((size - s) // 2, 0)                  # ... and where the pad starts in the output
    n = np.minimum(s, size)
    com = np.array(ndimage.center_of_mass(v))
    shift = (size - 1) / 2 - dst0 + src0 - com             # puts the centre of mass exactly at the output's centre
    v = ndimage.shift(v, shift, order=1)
    out = np.zeros((size, size, size), np.float32)
    out[dst0[0]:dst0[0] + n[0], dst0[1]:dst0[1] + n[1], dst0[2]:dst0[2] + n[2]] = \
        v[src0[0]:src0[0] + n[0], src0[1]:src0[1] + n[1], src0[2]:src0[2] + n[2]]
    out -= out.min()
    return out / (out.max() + 1e-12)


def random_rotation(rng: np.random.Generator) -> np.ndarray:
    """Uniformly random proper rotation (det = +1) from a random unit quaternion."""
    q = rng.normal(size=4)
    q /= np.linalg.norm(q)
    w, x, y, z = q
    return np.array([
        [1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
        [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
        [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])


_P = np.array([[0, 0, 1], [0, 1, 0], [1, 0, 0]], float)  # (x,y,z) <-> (z,y,x)
def rotate_volume(vol: np.ndarray, R: np.ndarray, shift_xyz=(0, 0, 0), order: int = 1,
                  cval: float = 0.0) -> np.ndarray:
    """Rotate a volume about its centre by R (then translate by shift, in voxels, (x,y,z)).

    out(p) = vol(Rᵀ (p − c − t) + c): the *object* is rotated by R and moved by t.
    """
    R_idx = _P @ R @ _P                       # same rotation written in (z, y, x) index order
    c = (np.array(vol.shape) - 1) / 2.0
    t_idx = _P @ np.asarray(shift_xyz, float)
    M = R_idx.T                               # inverse rotation maps output -> input
    offset = c - M @ (c + t_idx)
    return ndimage.affine_transform(vol, M, offset=offset, order=order, mode="constant", cval=cval)


def freq_grid(shape):
    """Frequency coordinates (cycles/voxel) for an unshifted FFT; returns list per axis."""
    return np.meshgrid(*[np.fft.fftfreq(n) for n in shape], indexing="ij")


def radial_frequency(shape) -> np.ndarray:
    """|k| in cycles/voxel for every coefficient of an unshifted FFT."""
    return np.sqrt(sum(k**2 for k in freq_grid(shape)))


def missing_wedge_mask(shape, tilt_max_deg: float = 60.0) -> np.ndarray:
    """Binary Fourier mask of a single-axis tilt series about y (±tilt_max).

    A coefficient (k_z, k_y, k_x) is measured if |k_z| ≤ |k_x|·tan(tilt_max)
    — the formula M(ξ) = 1(|ξ_3| ≤ |ξ_1| tan θ) used by Xu et al. (2012).
    For 2D images [z, x] pass a 2-tuple shape.
    """
    k = freq_grid(shape)
    kz, kx = k[0], k[-1]
    m = np.abs(kz) <= np.abs(kx) * np.tan(np.radians(tilt_max_deg)) + 1e-9
    m[(0,) * len(shape)] = True               # keep the mean (DC) term
    return m.astype(np.float32)


def apply_fourier_mask(vol: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Multiply the FFT of ``vol`` by ``mask`` (unshifted layout) and return the real part."""
    return np.real(np.fft.ifftn(np.fft.fftn(vol) * mask)).astype(np.float32)


def lowpass(vol: np.ndarray, cutoff: float, soft: float = 0.02) -> np.ndarray:
    """Soft low-pass filter; ``cutoff`` in cycles/voxel (0.5 = Nyquist)."""
    k = radial_frequency(vol.shape)
    filt = 1 / (1 + np.exp((k - cutoff) / soft))
    return apply_fourier_mask(vol, filt)


def add_noise_snr(signal: np.ndarray, snr: float, rng: np.random.Generator) -> np.ndarray:
    """Add white Gaussian noise so that var(signal)/var(noise) = snr."""
    sigma = np.sqrt(signal.var() / snr)
    return (signal + rng.normal(0, sigma, signal.shape)).astype(np.float32)


def normalize(v: np.ndarray) -> np.ndarray:
    """Zero mean, unit standard deviation."""
    return ((v - v.mean()) / (v.std() + 1e-12)).astype(np.float32)


def pearson(a: np.ndarray, b: np.ndarray) -> float:
    """Pearson correlation of two arrays (flattened)."""
    a = a.ravel() - a.mean()
    b = b.ravel() - b.mean()
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))


def electron_wavelength_A(kV: float = 300.0) -> float:
    """Relativistic electron wavelength in Å (0.0197 Å at 300 kV)."""
    V = kV * 1e3
    return 12.2643 / np.sqrt(V * (1 + 0.978466e-6 * V))


def ctf_2d(shape, pixel_A: float, defocus_um: float = 3.0, kV: float = 300.0,
           Cs_mm: float = 2.7, amp_contrast: float = 0.07, B: float = 0.0) -> np.ndarray:
    """Contrast transfer function on an unshifted FFT grid (underfocus is positive).

    CTF(k) = −[√(1−A²)·sin χ(k) + A·cos χ(k)]·exp(−B k²/4),
    χ(k) = π λ Δf k² − (π/2) C_s λ³ k⁴, with k in 1/Å.
    """
    lam = electron_wavelength_A(kV)
    k2 = sum(kk**2 for kk in freq_grid(shape)) / pixel_A**2
    dz = defocus_um * 1e4
    cs = Cs_mm * 1e7
    chi = np.pi * lam * dz * k2 - 0.5 * np.pi * cs * lam**3 * k2**2
    ctf = -(np.sqrt(1 - amp_contrast**2) * np.sin(chi) + amp_contrast * np.cos(chi))
    return (ctf * np.exp(-B * k2 / 4)).astype(np.float32)


def fsc(v1: np.ndarray, v2: np.ndarray, pixel_A: float = 10.0, n_shells: int | None = None):
    """Fourier shell (ring, in 2D) correlation. Returns (frequency_1/Å, fsc, n_coeffs)."""
    F1, F2 = np.fft.fftn(v1), np.fft.fftn(v2)
    k = radial_frequency(v1.shape)
    n_shells = n_shells or min(v1.shape) // 2
    idx = np.minimum((k * 2 * n_shells).astype(int), n_shells)
    num = np.bincount(idx.ravel(), (F1 * np.conj(F2)).real.ravel(), n_shells + 1)
    d1 = np.bincount(idx.ravel(), (np.abs(F1)**2).ravel(), n_shells + 1)
    d2 = np.bincount(idx.ravel(), (np.abs(F2)**2).ravel(), n_shells + 1)
    cnt = np.bincount(idx.ravel(), minlength=n_shells + 1)
    curve = num / np.sqrt(d1 * d2 + 1e-20)
    freq = (np.arange(n_shells + 1) + 0.5) / (2 * n_shells) / pixel_A
    return freq[:n_shells], curve[:n_shells], cnt[:n_shells]


def resolution_at(freq, curve, threshold: float = 0.143) -> float:
    """First crossing of the FSC curve below ``threshold`` → resolution in Å."""
    below = np.where(curve[1:] < threshold)[0]
    if len(below) == 0:
        return float(1 / freq[-1])
    i = below[0] + 1
    f0, f1, c0, c1 = freq[i - 1], freq[i], curve[i - 1], curve[i]
    f = f0 + (threshold - c0) * (f1 - f0) / (c1 - c0 + 1e-12)
    return float(1 / f)


def match_points(pred, true, radius: float):
    """Greedy one-to-one matching of predicted to true coordinates within ``radius``.

    Returns dict(precision, recall, f1, tp, fp, fn).
    """
    true = np.asarray(true, float)
    pred = np.asarray(pred, float).reshape(-1, true.shape[-1] if true.ndim == 2 else 3)
    if len(pred) == 0 or len(true) == 0:
        tp = 0
    else:
        d = np.linalg.norm(pred[:, None, :] - true[None, :, :], axis=-1)
        used_p, used_t, tp = set(), set(), 0
        for i, j in zip(*np.unravel_index(np.argsort(d, axis=None), d.shape)):
            if d[i, j] > radius:
                break
            if i not in used_p and j not in used_t:
                used_p.add(i); used_t.add(j); tp += 1
    fp, fn = len(pred) - tp, len(true) - tp
    p = tp / max(len(pred), 1)
    r = tp / max(len(true), 1)
    return dict(precision=p, recall=r, f1=2 * p * r / max(p + r, 1e-12), tp=tp, fp=fp, fn=fn)


def support_radius(t: np.ndarray, level: float = 0.1) -> float:
    """Radius (voxels) of the smallest centred sphere containing all voxels above ``level``·max."""
    pts = np.argwhere(t > level * t.max())
    if len(pts) == 0:
        return t.shape[0] / 2
    c = (np.array(t.shape) - 1) / 2
    return float(np.linalg.norm(pts - c, axis=1).max()) + 1.0


def make_crowded_cell(templates: dict, counts: dict, shape=(48, 160, 160), rng=None,
                      membrane: bool = True, tries_per_particle: int = 400):
    """Pack randomly rotated templates into a volume without overlap (bounding spheres).

    A simple rejection-sampling stand-in for the packing used by Pei et al. (2016)
    and Liu et al. (2020).  Particles are placed in random class order so that every
    class gets a fair chance.  Returns (clean_volume, particles) where each particle
    is dict(cls, center_zyx, R).  Optionally adds a spherical membrane vesicle.
    """
    rng = rng or np.random.default_rng(0)
    vol = np.zeros(shape, np.float32)
    occ_c, occ_r, particles = [], [], []
    if membrane:  # a vesicle: thin shell, ~4 nm thick lipid bilayer at 1 nm/voxel
        zc, yc, xc = shape[0] / 2, shape[1] * 0.78, shape[2] * 0.22
        rad = min(shape[1], shape[2]) * 0.16
        g = np.meshgrid(*[np.arange(n) for n in shape], indexing="ij")
        d = np.sqrt((g[0] - zc)**2 + (g[1] - yc)**2 + (g[2] - xc)**2)
        vol += 0.6 * np.exp(-((d - rad)**2) / (2 * 1.5**2))
        occ_c.append(np.array([zc, yc, xc])); occ_r.append(rad + 3)
    radii = {cls: support_radius(templates[cls], level=0.35) for cls in counts}
    order = [cls for cls, n in counts.items() for _ in range(n)]
    rng.shuffle(order)
    for cls in order:
        t, rad_t = templates[cls], radii[cls]
        for _ in range(tries_per_particle):
            c = np.array([rng.uniform(rad_t, s - rad_t) if s > 2 * rad_t else s / 2 for s in shape])
            if any(np.linalg.norm(c - oc) < rad_t + orr for oc, orr in zip(occ_c, occ_r)):
                continue
            R = random_rotation(rng)
            sub = rotate_volume(t, R)
            h = np.array(sub.shape) // 2
            lo = np.round(c).astype(int) - h
            sl_v, sl_s = [], []
            for a in range(3):
                a0, a1 = max(lo[a], 0), min(lo[a] + sub.shape[a], shape[a])
                sl_v.append(slice(a0, a1)); sl_s.append(slice(a0 - lo[a], a1 - lo[a]))
            vol[tuple(sl_v)] += sub[tuple(sl_s)]
            occ_c.append(c); occ_r.append(rad_t)
            particles.append(dict(cls=cls, center_zyx=c, R=R))
            break
    return vol, particles


def simulate_tomogram(clean: np.ndarray, rng, snr: float = 0.05, tilt_max_deg: float = 60.0,
                      ctf_pixel_A: float | None = 10.0, defocus_um: float = 5.0) -> np.ndarray:
    """Fast tomogram simulation: (optional 3D CTF-like filter) → missing wedge → noise.

    The CTF is applied isotropically here (a simplification of per-tilt CTF).
    """
    v = clean
    if ctf_pixel_A:
        ctf = ctf_2d(clean.shape, ctf_pixel_A, defocus_um=defocus_um, kV=300, B=0)
        v = apply_fourier_mask(v, -ctf)          # sign flip: dense objects appear bright
    v = apply_fourier_mask(v, missing_wedge_mask(v.shape, tilt_max_deg))
    return add_noise_snr(v, snr, rng)
print("🧰 toolkit loaded")

---
## 1 · The sealed sample

📍 **Where we are.** Everything from Module 00 onwards comes together here. The sample's composition is stored in a
sealed (base64-encoded) string: the code decodes it to build the cell, but **you should not print it** until Section 11.
The imaging parameters, on the other hand, are known to you — just as a microscopist knows the voxel size, the tilt
range and the defocus.

In [ ]:
# ── 1.1 Build the cell from the sealed composition and record a tomogram ──
SEALED = "eyI0djRyIjogMzAsICIzajlpIjogMzUsICIxYnhuIjogNDUsICIzY2YzIjogMzB9"   # 🔒 do not decode before Section 11
_composition = json.loads(base64.b64decode(SEALED))
SHAPE = (64, 224, 224)                    # z, y, x voxels of 10 Å → a 64 × 224 × 224 nm slab of cytoplasm
VOXEL_A, TILT_MAX, DEFOCUS_UM, SNR = 10.0, 60.0, 5.0, 0.3
t0 = time.time()
_templates = {k: prepare_map(zoo_map(k)[0], size=32) for k in _composition}
_clean, _truth = make_crowded_cell(_templates, _composition, shape=SHAPE, rng=rng, membrane=False)
tomogram = simulate_tomogram(_clean, rng, snr=SNR, tilt_max_deg=TILT_MAX, ctf_pixel_A=VOXEL_A, defocus_um=DEFOCUS_UM)
print(f"tomogram {tomogram.shape} recorded in {time.time() - t0:.1f} s "
      f"(voxel {VOXEL_A:.0f} Å, tilt ±{TILT_MAX:.0f}°, defocus {DEFOCUS_UM} µm, SNR {SNR})")
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), gridspec_kw=dict(width_ratios=[224, 224]))
axes[0].imshow(tomogram[SHAPE[0] // 2]); axes[0].set_title("central xy slice")
axes[1].imshow(tomogram[:, SHAPE[1] // 2], aspect=2); axes[1].set_title("central xz slice (note the elongation along z)")
for a in axes:
    a.axis("off")
plt.tight_layout(); plt.show()

---
## 2 · Picking: difference of Gaussians and an elbow

Template-free picking (Module 11; Pei *et al.* 2016): smooth at two scales, subtract, and keep local maxima. Large
particles produce several maxima, so we suppress peaks closer than 7 voxels to a stronger one. How many to keep? We look
for the **elbow** of the sorted peak scores — where they stop falling steeply.

In [ ]:
# ── 2.1 DoG picking ──
SIGMA_DOG = 3.5
dog = ndimage.gaussian_filter(tomogram, SIGMA_DOG) - ndimage.gaussian_filter(tomogram, 1.6 * SIGMA_DOG)
peaks = np.argwhere(dog == ndimage.maximum_filter(dog, size=15))
scores = dog[tuple(peaks.T)]
order = np.argsort(-scores)
peaks, scores = peaks[order], scores[order]
top = scores[:400]
chord = top[0] + (top[-1] - top[0]) * np.arange(len(top)) / (len(top) - 1)
n_picks = int(np.argmax(chord - top))                               # farthest point below the chord = elbow
picks = peaks[:n_picks]
print(f"{len(peaks)} local maxima; elbow at {n_picks} → we keep {n_picks} picks")
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
axes[0].plot(top, "k-"); axes[0].plot(chord, "c--", lw=1); axes[0].axvline(n_picks, color="tab:red")
axes[0].set_xlabel("peak rank"); axes[0].set_ylabel("DoG score"); axes[0].set_title("sorted peak scores and the elbow")
z0 = SHAPE[0] // 2
near = np.abs(picks[:, 0] - z0) <= 4
axes[1].imshow(tomogram[z0]); axes[1].scatter(picks[near, 2], picks[near, 1], s=60, facecolors="none", edgecolors="tab:orange")
axes[1].set_title("picks within ±4 slices of the centre"); axes[1].axis("off")
plt.tight_layout(); plt.show()

---
## 3 · Subtomograms and rotation-invariant features

We cut a 32³ box (32 nm) around each pick and **recentre** it on its density (three rounds, re-cutting the box each
time). Recentring has a bonus: two picks on the same large particle drift to the same centre, so we can **merge
duplicates** (keeping the stronger pick). Finally we remove the noise that lies inside the missing wedge (Module 07:
those Fourier coefficients carry no signal). To compare particles in *unknown* orientations,
the first clustering uses **rotation-invariant** features (Module 15): the radial density profile, the spread of the
density along its three principal axes, the mass within four radii, and the radial power spectrum.

In [ ]:
# ── 3.1 Extraction, recentring and wedge filtering ──
BOX = 32
padded = np.pad(normalize(tomogram), BOX // 2)
grid = np.indices((BOX,) * 3).astype(np.float32) - (BOX / 2 - 0.5)
radius = np.sqrt((grid ** 2).sum(0))
WEDGE = missing_wedge_mask((BOX,) * 3, TILT_MAX)
KFREQ = radial_frequency((BOX,) * 3)


def extract(centre):
    c = np.round(centre).astype(int) + BOX // 2
    return padded[c[0] - BOX // 2:c[0] + BOX // 2, c[1] - BOX // 2:c[1] + BOX // 2, c[2] - BOX // 2:c[2] + BOX // 2]


def recentre(v):
    """Shift the box so that the centre of the (smoothed, positive) density sits in the middle; return box and shift."""
    w = np.clip(ndimage.gaussian_filter(v, 2), 0, None) * (radius < 12)
    com = (grid * w).sum((1, 2, 3)) / (w.sum() + 1e-9)
    return ndimage.shift(v, -com, order=1, mode="nearest"), com


def wedge_filter(v):
    return np.real(np.fft.ifftn(np.fft.fftn(v) * WEDGE)).astype(np.float32)


centres = picks.astype(float)
for _ in range(3):                                                  # recentre, re-cut, repeat
    centres = centres + np.array([recentre(extract(c))[1] for c in centres])
keep = []                                                           # merge duplicates (picks are sorted by score)
for i, c in enumerate(centres):
    if all(np.linalg.norm(c - centres[j]) > 6 for j in keep):
        keep.append(i)
centres = centres[keep]
boxes = [recentre(extract(c))[0] for c in centres]
subtomos = np.stack([wedge_filter(b) for b in boxes])
print(f"{len(picks)} picks → {len(centres)} after recentring and merging duplicates → {len(subtomos)} subtomograms of {BOX}³ voxels")

In [ ]:
# ── 3.2 Rotation-invariant features ──
SHELL = np.minimum(radius.astype(int), 16)
KSHELL = np.minimum((KFREQ * BOX).astype(int), 16)


def invariant_features(v):
    """Radial profile (16) + principal spreads (3) + mass in 4 spheres (4) + log radial power (12)."""
    lp = ndimage.gaussian_filter(v, 1.5)
    profile = np.bincount(SHELL.ravel(), lp.ravel(), 17)[:16] / np.bincount(SHELL.ravel(), minlength=17)[:16]
    w = np.clip(lp, 0, None) * (radius < 15)
    cov = np.einsum("iabc,jabc->ij", grid * w, grid) / (w.sum() + 1e-9)
    spreads = np.sqrt(np.clip(np.sort(np.linalg.eigvalsh(cov)), 0, None))
    mass = [(lp * (radius < r)).sum() / 100 for r in (6, 9, 12, 15)]
    power = np.log(np.bincount(KSHELL.ravel(), (np.abs(np.fft.fftn(v * (radius < 15))) ** 2).ravel(), 17)[1:13])
    return np.concatenate([profile, spreads, mass, power])


feats = np.stack([invariant_features(b) for b in boxes])
feats = (feats - feats.mean(0)) / (feats.std(0) + 1e-9)
print("feature matrix:", feats.shape)

---
## 4 · Template-free classification: how many kinds of particles?

As in DISCA (Module 22), we reduce the features with PCA and fit Gaussian mixture models for a range of $K$; the
**Bayesian information criterion** $\mathrm{BIC}=p\ln n-2\ln\hat L$ (Module 04) chooses $K$, balancing fit against the
number of parameters $p$. *(DISCA learns its features with a neural network and iterates; our hand-crafted invariant
features are a fast baseline — replacing them is a suggested project.)*

In [ ]:
# ── 4.1 PCA + GMM with K chosen by BIC ──
def gmm_bic(X, ks, seed=0):
    """Fit GMMs for each K; return (best K, labels, list of BIC values)."""
    X = np.asarray(X, np.float64)
    ks = [k for k in ks if len(X) >= 10 * k]                       # at least ~10 particles per component
    models = [GaussianMixture(k, random_state=seed, n_init=5, reg_covar=1e-4).fit(X) for k in ks]
    bics = [m.bic(X) for m in models]
    i = int(np.argmin(bics))
    return ks[i], models[i].predict(X), bics


Z = PCA(6, random_state=0).fit_transform(feats)
K1, level1, bics = gmm_bic(Z, list(range(2, 9)))
MIN_CLASS = 10
sizes = collections.Counter(level1)
print(f"BIC chooses K = {K1}; cluster sizes: {dict(sorted(sizes.items()))}")
print(f"clusters with fewer than {MIN_CLASS} members are set aside as 'unassigned'")
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].plot(range(2, 9), bics, "o-"); axes[0].axvline(K1, color="tab:red"); axes[0].set_xlabel("K"); axes[0].set_ylabel("BIC")
axes[1].scatter(Z[:, 0], Z[:, 1], c=level1, cmap="tab10", s=12); axes[1].set_xlabel("PC 1"); axes[1].set_ylabel("PC 2")
axes[1].set_title("picks in feature space, coloured by cluster")
plt.tight_layout(); plt.show()

---
## 5 · Alignment and averaging inside each cluster

Within a cluster we align every subtomogram to a common reference and average (Modules 12–14). Three ingredients:

* **Wedge-aware scoring.** All subtomograms come from one tomogram, so they share one missing wedge $M$. We compare a
  subtomogram $s$ with the rotated reference only where data exist and within a resolution band:
  $\ \mathrm{score}(R,\mathbf t)=\sum_{\mathbf k} \hat s(\mathbf k)\,\overline{M(\mathbf k)\,\widehat{R\,r}(\mathbf k)}\,e^{2\pi i\mathbf k\cdot\mathbf t}\big/\lVert M\,\widehat{R\,r}\rVert$
  — for every translation at once by FFT, for 400 random rotations.
* **Wedge-weighted averaging.** After rotating particle $i$ back by $R_i^{-1}$, its measured coefficients sit on the
  rotated wedge $M_i(\boldsymbol\xi)=M(R_i\boldsymbol\xi)$; each Fourier coefficient of the average is divided by the number
  of particles that measured it.
* **Reference-free start.** The first reference is the cluster's mean, heavily low-passed; every round sharpens it.

In [ ]:
# ── 5.1 The alignment engine ──
MAX_SHIFT = 3
_sh = np.indices((BOX,) * 3)
_sh = np.where(_sh > BOX // 2, _sh - BOX, _sh)                       # FFT index → signed shift
SHIFT_WINDOW = np.flatnonzero((np.abs(_sh) <= MAX_SHIFT).all(0).ravel())
SHIFTS = _sh.reshape(3, -1)[:, SHIFT_WINDOW].T
rot_rng = np.random.default_rng(3)
ROTATIONS = [random_rotation(rot_rng) for _ in range(400)]


def rotated_wedge(R):
    """M_i(ξ) = M(R ξ): the coefficients a particle with rotation R measured, in the reference frame."""
    kz, ky, kx = freq_grid((BOX,) * 3)
    kr = np.tensordot(R, np.stack([kx, ky, kz]), axes=1)
    m = np.abs(kr[2]) <= np.abs(kr[0]) * np.tan(np.radians(TILT_MAX)) + 1e-9
    m[0, 0, 0] = True
    return m.astype(np.float32)


def align(X, ref, band=0.25):
    """Best (score, R, shift_zyx) of every subtomogram in X against `ref` over ROTATIONS × small shifts."""
    FX = sfft.rfftn(X.astype(np.float32), axes=(1, 2, 3), workers=-1)
    filt = (WEDGE * (KFREQ < band) * (KFREQ > 0))[:, :, :BOX // 2 + 1]
    best, best_R, best_t = np.full(len(X), -np.inf), [None] * len(X), np.zeros((len(X), 3), int)
    for R in ROTATIONS:
        Fr = (sfft.rfftn(rotate_volume(ref, R)) * filt).astype(np.complex64)
        Fr /= np.sqrt((np.abs(Fr) ** 2).sum()) + 1e-12
        cc = sfft.irfftn(FX * np.conj(Fr)[None], s=(BOX,) * 3, axes=(1, 2, 3), workers=-1).reshape(len(X), -1)[:, SHIFT_WINDOW]
        j = cc.argmax(1)
        v = cc[np.arange(len(X)), j]
        for n in np.flatnonzero(v > best):
            best[n], best_R[n], best_t[n] = v[n], R, SHIFTS[j[n]]
    return best, best_R, best_t


def to_reference_frame(x, R, t):
    """Undo the shift and the rotation found by `align`."""
    return rotate_volume(ndimage.shift(x, -t, order=1), R.T)


def wedge_average(X, Rs, ts):
    """Missing-wedge-weighted average of aligned particles."""
    num, den = np.zeros((BOX,) * 3, complex), np.zeros((BOX,) * 3)
    for x, R, t in zip(X, Rs, ts):
        m = rotated_wedge(R)
        num += np.fft.fftn(to_reference_frame(x, R, t)) * m
        den += m
    return np.real(np.fft.ifftn(np.where(den > 0, num / np.maximum(den, 1), 0))).astype(np.float32)


def refine(X, ref, n_iter=3, band=0.25):
    """Iterative alignment + averaging; returns (average, rotations, shifts, scores)."""
    for _ in range(n_iter):
        sc, Rs, ts = align(X, ref, band)
        ref = lowpass(wedge_average(X, Rs, ts), band)
    return ref, Rs, ts, sc


print(f"alignment engine ready: {len(ROTATIONS)} rotations × {len(SHIFTS)} shifts per particle")

In [ ]:
# ── 5.2 Reference-free refinement of each cluster ──
t0 = time.time()
clusters = {}
for c in sorted(sizes):
    idx = np.flatnonzero(level1 == c)
    if len(idx) < MIN_CLASS:
        continue
    avg, Rs, ts, sc = refine(subtomos[idx], lowpass(subtomos[idx].mean(0), 0.12))
    clusters[c] = dict(idx=idx, avg=avg, R=Rs, t=ts, score=sc)
    print(f"cluster {c}: {len(idx):3d} particles, mean alignment score {sc.mean():.2f}  ({time.time() - t0:.0f} s)")

fig, axes = plt.subplots(1, len(clusters), figsize=(3 * len(clusters), 3))
for ax, (c, d) in zip(np.atleast_1d(axes), clusters.items()):
    ax.imshow(d["avg"].sum(0)); ax.set_title(f"cluster {c} (n = {len(d['idx'])})"); ax.axis("off")
plt.suptitle("cluster averages (projected along z)", fontsize=9); plt.tight_layout(); plt.show()

---
## 6 · Looking inside each cluster

Rotation-invariant features can miss differences between molecules of similar size. Now that particles are aligned we
can compare them **voxel by voxel**. Each aligned particle misses a different wedge of information, which would make
particles look different for the wrong reason, so we **impute** the missing coefficients from the cluster average
(as Gum-Net does, Module 21):

$$\hat x_i^{\text{imputed}}(\boldsymbol\xi)=M_i(\boldsymbol\xi)\,\hat x_i(\boldsymbol\xi)+\big(1-M_i(\boldsymbol\xi)\big)\,\hat a(\boldsymbol\xi).$$

Then PCA inside a mask, and a GMM with $K\in\{1,2,3\}$: BIC decides whether the cluster should be split.

In [ ]:
# ── 6.1 Imputation + PCA + BIC inside each cluster ──
classes = []                                                        # final classes: (indices, rotations, shifts)
for c, d in clusters.items():
    X, Fa = subtomos[d["idx"]], np.fft.fftn(d["avg"])
    mask = ndimage.binary_dilation(ndimage.gaussian_filter(d["avg"], 1.5) > 0.3 * d["avg"].max(), iterations=2)
    imputed = []
    for x, R, t in zip(X, d["R"], d["t"]):
        m = rotated_wedge(R)
        v = np.real(np.fft.ifftn(m * np.fft.fftn(to_reference_frame(x, R, t)) + (1 - m) * Fa))
        imputed.append(lowpass(v, 0.2)[mask])
    imputed = np.array(imputed) - np.mean(imputed, 0)
    Zc = PCA(4, random_state=0).fit_transform(imputed)
    k2, sub, b2 = gmm_bic(Zc, [1, 2, 3])
    print(f"cluster {c}: BIC for K = 1, 2, 3 → {np.round(b2, 0)} ⇒ {'keep as one class' if k2 == 1 else f'split into {k2}'}")
    for s in range(k2):
        keep = sub == s
        if keep.sum() >= MIN_CLASS:
            classes.append(dict(idx=d["idx"][keep], R=[r for r, k in zip(d["R"], keep) if k], t=d["t"][keep], parent=c))
        else:
            print(f"   (a sub-cluster of {keep.sum()} particles is set aside as unassigned)")
print(f"→ {len(classes)} final classes with sizes {[len(cl['idx']) for cl in classes]}")

🔍 **What just happened?** BIC looked inside each cluster. Splitting a cluster that mixes two molecules of similar size
is exactly what this step is for. But BIC can also split a cluster of **one** molecule — for example by alignment
quality or by how well particles were centred. Such over-splitting is not fatal: it shows up later as two classes with
the same identity, which you should then test for merging (✏️ below, and the redundancy test of MPP in Module 15).

✏️ **Your turn (after Section 8).** If two classes get the same identity, align one average onto the other with
`align(avg_b[None], avg_a)` and `to_reference_frame`, and compare their correlation with each class's half-map
correlation `r_half`. If the two classes agree as well as each agrees with itself, merge them and recompute the FSC.

---
## 7 · Gold-standard resolution

The **Fourier shell correlation** between two *independent* half-maps (Module 10) measures how reproducible a map is
at each resolution. "Independent" matters: if both halves are aligned to one shared reference, noise aligned to the same
features inflates the FSC. So we split every class at random into two halves, refine each half **separately** from a
very blurry common start (Ø ≈ 80 Å detail only), align half 2 onto half 1 at the end, and compute the FSC within a soft
spherical mask. The resolution is where the curve drops below 0.143.

In [ ]:
# ── 7.1 Independent half refinements and FSC ──
SOFT_MASK = ndimage.gaussian_filter((radius < 14).astype(np.float32), 1.5)
t0 = time.time()
for k, cl in enumerate(classes):
    X = subtomos[cl["idx"]]
    start = lowpass(wedge_average(X, cl["R"], cl["t"]), 0.125)       # shared start: only ≥ 80 Å detail
    halves = np.random.default_rng(k).permutation(len(X)) % 2
    h = [refine(X[halves == i], start, n_iter=2)[0] for i in (0, 1)]
    _, R12, t12 = align(h[1][None], h[0])                           # bring half 2 onto half 1
    h2 = to_reference_frame(h[1], R12[0], t12[0])
    freq, curve, _ = fsc(h[0] * SOFT_MASK, h2 * SOFT_MASK, pixel_A=VOXEL_A)
    cl.update(half1=h[0], half2=h2, average=(h[0] + h2) / 2, fsc=(freq, curve),
              resolution=resolution_at(freq, curve, 0.143), r_half=pearson(h[0] * SOFT_MASK, h2 * SOFT_MASK))
    print(f"class {k}: {len(X):3d} particles, resolution (FSC 0.143) ≈ {cl['resolution']:.0f} Å  ({time.time() - t0:.0f} s)")

fig, ax = plt.subplots(figsize=(6.4, 3.2))
for k, cl in enumerate(classes):
    ax.plot(cl["fsc"][0][1:], cl["fsc"][1][1:], label=f"class {k} (n = {len(cl['idx'])})")      # skip the mean (DC) shell
ax.axhline(0.143, color="k", ls="--", lw=0.8); ax.set_xlabel("spatial frequency (1/Å)"); ax.set_ylabel("FSC")
ax.set_ylim(-0.2, 1.05); ax.legend(fontsize=8); plt.show()

---
## 8 · Identification — and discovery

We compare each class average with a **library** of known structures, filtered by the same CTF as the data and searched
over rotations and shifts. But a best match always exists; is it *good enough*? A useful yardstick is the map's own
reproducibility. If the two half-maps correlate with each other by $r_{1/2}$, the full map should correlate with the
*true* structure by about

$$C_{\text{expected}}=\sqrt{\frac{2\,r_{1/2}}{1+r_{1/2}}}$$

(the same relation that links FSC to the map–truth correlation, Rosenthal & Henderson 2003). A library match far below
$C_{\text{expected}}$ means "the right structure is probably not in the library" — a candidate **discovery**.

In [ ]:
# ── 8.1 Library search ──
LIBRARY = ["4v4r", "3j9i", "6utj", "1bxn", "1qvr", "4cr2"]          # what we would try first for a bacterium-like cell
CTF3D = ctf_2d((BOX,) * 3, VOXEL_A, defocus_um=DEFOCUS_UM)
library = {pid: apply_fourier_mask(prepare_map(zoo_map(pid)[0], size=BOX), -CTF3D) for pid in LIBRARY}


def unit(v, band=0.25):
    """Band-limit, mask, centre and scale a map to unit length (so dot products are correlations)."""
    v = lowpass(v, band) * SOFT_MASK
    v = v - v.mean()
    return v / (np.linalg.norm(v) + 1e-12)


def small_rotation(rng, max_deg):
    """A random rotation by at most `max_deg` degrees about a random axis (Rodrigues' formula)."""
    axis = rng.normal(size=3); axis /= np.linalg.norm(axis)
    a = np.radians(rng.uniform(0, max_deg))
    K = np.array([[0, -axis[2], axis[1]], [axis[2], 0, -axis[0]], [-axis[1], axis[0], 0]])
    return np.eye(3) + np.sin(a) * K + (1 - np.cos(a)) * K @ K


def match_score(Fa, ref, R):
    """Correlation of a unit-normalised average (given by its FFT) with `ref` rotated by R, best over small shifts."""
    cc = np.real(np.fft.ifftn(Fa * np.conj(np.fft.fftn(unit(rotate_volume(ref, R))))))
    return float(cc.ravel()[SHIFT_WINDOW].max())


def library_scores(averages, library, rotations=ROTATIONS[:200], seed=0):
    """Best correlation of every average with every library map: coarse search over `rotations`, then local refinement
    (random perturbations of the best rotation by ≤ 15°, 8° and 4°), so asymmetric molecules are not penalised."""
    r_loc = np.random.default_rng(seed)
    FA = np.stack([np.fft.fftn(unit(a)) for a in averages])
    out = np.zeros((len(averages), len(library)))
    for j, ref in enumerate(library.values()):
        coarse = np.full(len(averages), -np.inf)
        coarse_R = [None] * len(averages)
        for R in rotations:                                         # coarse search, all averages at once
            Fr = np.conj(np.fft.fftn(unit(rotate_volume(ref, R))))
            cc = np.real(np.fft.ifftn(FA * Fr[None], axes=(1, 2, 3))).reshape(len(averages), -1)[:, SHIFT_WINDOW].max(1)
            for i in np.flatnonzero(cc > coarse):
                coarse[i], coarse_R[i] = cc[i], R
        for i, Fa in enumerate(FA):                                 # local refinement, one average at a time
            best, best_R = coarse[i], coarse_R[i]
            for step in (15, 8, 4):
                for _ in range(15):
                    R = small_rotation(r_loc, step) @ best_R
                    sc = match_score(Fa, ref, R)
                    if sc > best:
                        best, best_R = sc, R
            out[i, j] = best
    return out


t0 = time.time()
S_lib = library_scores([cl["average"] for cl in classes], library)
for k, cl in enumerate(classes):
    cl["matches"] = dict(zip(LIBRARY, S_lib[k]))
    cl["expected"] = np.sqrt(2 * cl["r_half"] / (1 + cl["r_half"]))
    top = max(cl["matches"], key=cl["matches"].get)
    cl["identity"] = top if cl["matches"][top] > cl["expected"] - 0.15 else "UNKNOWN"
    ranked = sorted(cl["matches"].items(), key=lambda kv: -kv[1])[:3]
    print(f"class {k}: expected ≈ {cl['expected']:.2f} | best matches " +
          ", ".join(f"{p.upper()} {v:.2f}" for p, v in ranked) + f"  ⇒ {cl['identity'].upper()}")
print(f"({time.time() - t0:.0f} s)")

🔍 **What just happened?** Classes whose averages correlate with a library entry about as well as their own
reproducibility allows are identified. A class whose best match falls well short is flagged **UNKNOWN**: the map is
reproducible, but nothing in the library explains it. In a real study that is where the exciting work begins — higher
resolution, fitting candidate structures predicted by AlphaFold, mass spectrometry of the sample, or correlative light
microscopy. *(The margin of 0.15 is our choice for this toy; with real data you would calibrate it, e.g. on classes of
known identity.)*

---
## 9 · Spatial statistics

Two questions from Module 33: **how much** of each class (concentration), and **where** — is any class spatially
segregated from the others? For the second we use a **permutation test**: shuffle class labels among all assigned
particle positions many times and ask whether a class's mean nearest-neighbour distance to its *own* kind is unusually
small (clustering) or large (dispersion) compared with the shuffles.

In [ ]:
# ── 9.1 Concentrations and a segregation test ──
N_AV = 6.022e23
volume_um3 = np.prod(SHAPE) * (VOXEL_A / 1e4) ** 3
positions = np.vstack([centres[cl["idx"]] for cl in classes]) * VOXEL_A / 10          # nm
labels = np.concatenate([[k] * len(cl["idx"]) for k, cl in enumerate(classes)])


def own_kind_nnd(pos, lab, k):
    p = pos[lab == k]
    return cKDTree(p).query(p, k=2)[0][:, 1].mean()


perm_rng = np.random.default_rng(9)
for k, cl in enumerate(classes):
    obs = own_kind_nnd(positions, labels, k)
    null = np.array([own_kind_nnd(positions, perm_rng.permutation(labels), k) for _ in range(500)])
    p_two = 2 * min((null <= obs).mean(), (null >= obs).mean())
    cl["conc"] = len(cl["idx"]) / (N_AV * volume_um3 * 1e-15) * 1e6
    cl["seg_p"] = min(1.0, p_two + 1 / 500)
    print(f"class {k} ({cl['identity'].upper():7s}): {len(cl['idx']):3d} copies ≈ {cl['conc']:5.1f} µM | "
          f"mean NND to own kind {obs:5.1f} nm (shuffled: {null.mean():5.1f} nm), permutation p ≈ {cl['seg_p']:.2f}")

🔍 **What just happened? — read the p-values with suspicion.** Small molecules often come out "clustered" with their own
kind (small p), and large ones not. Before calling this biology, ask what the null model assumed: shuffling labels among
the observed positions ignores **excluded volume** — two small molecules can sit closer together than a small one and a
ribosome can. So small classes look clustered even in a randomly packed cell. A fair null would re-pack molecules of the
same sizes and numbers at random (exactly what `make_crowded_cell` does) and compare with that (✏️ Bonus). This is the
Module 33 lesson again: **the null model is a scientific choice.**

---
## 10 · The report

A good report shows the evidence for every claim: the averages, their resolution, how they were identified, and how
many there are — with the methods and parameters needed to reproduce them.

In [ ]:
# ── 10.1 Results table and figure ──
print(f"{'class':>5s} {'n':>4s} {'identity':>9s} {'best match':>11s} {'expected':>9s} {'resolution':>11s} {'conc. (µM)':>10s} {'segregation p':>14s}")
for k, cl in enumerate(classes):
    print(f"{k:5d} {len(cl['idx']):4d} {cl['identity'].upper():>9s} {max(cl['matches'].values()):11.2f} {cl['expected']:9.2f} "
          f"{cl['resolution']:9.0f} Å {cl['conc']:10.1f} {cl['seg_p']:14.2f}")
n_unassigned = len(centres) - sum(len(cl["idx"]) for cl in classes)
print(f"unassigned particles: {n_unassigned} of {len(centres)}")

fig, axes = plt.subplots(2, len(classes), figsize=(2.6 * len(classes), 5.2))
for k, cl in enumerate(classes):
    av = cl["average"]
    axes[0, k].imshow(av.sum(0)); axes[0, k].set_title(f"class {k}: {cl['identity'].upper()}\nn = {len(cl['idx'])}, "
                                                        f"{cl['resolution']:.0f} Å", fontsize=8)
    axes[1, k].imshow(av.sum(1)); axes[1, k].set_title("side view (xz)", fontsize=8)
    for a in axes[:, k]:
        a.axis("off")
plt.tight_layout(); plt.show()
print(f"⏱ pipeline so far: {time.time() - T_START:.0f} s")

✏️ **Your turn — write the abstract.** In five sentences: what was imaged, what you found (how many classes, which
identities, one unknown?), how confident you are in each claim (resolution, match quality), and what you would do next.
Write it **before** you unseal the truth.

---
## 11 · Unsealing: evaluation against the hidden truth

Only now do we open the envelope. Evaluating at the end — never tuning on the answer — is what keeps a benchmark honest.

In [ ]:
# ── 11.1 The truth ──
print("sealed composition:", _composition)
true_xyz = np.array([p["center_zyx"] for p in _truth])
true_cls = np.array([p["cls"] for p in _truth])
print("molecules actually placed:", dict(collections.Counter(true_cls)))
in_library = {pid: (pid in LIBRARY) for pid in _composition}
print("in our library?", in_library)

In [ ]:
# ── 11.2 Picking, classification and identification scores ──
for name, pts in [("raw DoG picks", picks), ("after recentring + merging", centres)]:
    m = match_points(pts, true_xyz, 7)
    print(f"picking, {name:26s}: precision {m['precision']:.2f}, recall {m['recall']:.2f}, F1 {m['f1']:.2f}")
d = np.linalg.norm(centres[:, None] - true_xyz[None], axis=-1)
pick_truth = np.where(d.min(1) < 7, true_cls[d.argmin(1)], "false pick")
pred = np.full(len(centres), -1)
for k, cl in enumerate(classes):
    pred[cl["idx"]] = k
print(f"classification: adjusted Rand index (all picks, unassigned = own group) = {adjusted_rand_score(pick_truth, pred):.2f}")
for k, cl in enumerate(classes):
    content = collections.Counter(pick_truth[cl["idx"]])
    majority, n_major = content.most_common(1)[0]
    verdict = "✅" if (cl["identity"] == majority) or (cl["identity"] == "UNKNOWN" and majority not in LIBRARY) else "❌"
    print(f"class {k}: called {cl['identity'].upper():7s} | truth {dict(content)} → purity {n_major / len(cl['idx']):.2f} {verdict}")
print("unassigned picks were:", dict(collections.Counter(pick_truth[pred < 0])))

In [ ]:
# ── 11.3 Map vs truth: how good are the averages really? ──
fig, ax = plt.subplots(figsize=(6.4, 3.2))
for k, cl in enumerate(classes):
    majority = collections.Counter(pick_truth[cl["idx"]]).most_common(1)[0][0]
    if majority == "false pick":
        continue
    truth_map = apply_fourier_mask(_templates[majority], -CTF3D)
    _, Rt, tt = align(truth_map[None], cl["average"])                 # put the true structure in the class frame
    truth_in_frame = to_reference_frame(truth_map, Rt[0], tt[0])
    freq, curve, _ = fsc(cl["average"] * SOFT_MASK, truth_in_frame * SOFT_MASK, pixel_A=VOXEL_A)
    ax.plot(freq[1:], curve[1:], label=f"class {k} vs {majority.upper()}: {resolution_at(freq, curve, 0.5):.0f} Å at 0.5")
ax.axhline(0.5, color="k", ls="--", lw=0.8); ax.set_xlabel("spatial frequency (1/Å)"); ax.set_ylabel("FSC to truth")
ax.set_ylim(-0.2, 1.05); ax.legend(fontsize=8); ax.set_title("class averages compared with the true structures"); plt.show()
print(f"⏱ whole capstone: {time.time() - T_START:.0f} s")

🔍 **What just happened?** Compare the evaluation with the abstract you wrote. Things to look for:

* **Composition.** Compare the sealed recipe with the molecules actually placed: if the packer could not fit every
  copy into the slab (large ribosomes are the first to be dropped, Module 09), the sample holds fewer than the recipe
  asked for. Concentrations must be checked against what is really there.
* **Picking.** Raw DoG picks include second peaks on large particles; recentring and merging duplicates removes most of
  them, which is why the precision rises between the two lines of 11.2.
* **Two-level classification.** Rotation-invariant features separate molecules of different size; the aligned, imputed
  voxel comparison separates similar-sized ones — including the molecule that was not in the library. Check the purity
  of each class: where did the mistakes go?
* **Over-splitting.** If two classes received the same identity, the truth tells you whether BIC split one molecule in
  two. Would your merge test (✏️ in Section 6) have caught it?
* **Resolution.** The half-map FSC at 0.143 and the FSC to the truth at 0.5 should roughly agree. Where they disagree —
  typically for the smallest classes — the half-map estimate is optimistic: few particles, a shared starting map and a
  mask all inflate FSC. That is why papers report particle numbers, masks and the FSC criterion.
* **Discovery.** The UNKNOWN class(es) were reproducible and clearly worse matched than the others — and they were indeed
  the molecule missing from the library. If BIC split it into two classes, would your merge test have reunited them?
  Notice that the library contained another AAA+ machine (ClpB) that did *not* capture it. With fewer copies, a lower
  SNR or a closer look-alike in the library, it could easily have been mislabelled.
* **Spatial statistics.** The cell was packed **at random**, so every "segregation" signal in Section 9 is an artefact of
  the null model — a clean demonstration of why null models matter.

✏️ **Your turn.** Re-run the whole notebook with `SNR = 0.1`. Which step breaks first? Then change the seed and compare
— conclusions that depend on one seed are anecdotes.

---
## 12 · Paper audit: reading like a reviewer

Every lab paper you read in this course can be audited with the same eight questions. Here is the worksheet filled in for
**DISCA** (Zeng, Kahng, Xue, Mahamid, Chang & Xu, PNAS 2023), using what Module 22 established; items we could not verify
are marked ⚠️ — finding them is part of the exercise.

| Question | DISCA (PNAS 2023) |
|---|---|
| **Biological question** | Which structurally distinct complexes and features are present in cellular tomograms, found without templates or labels? |
| **Data & split unit** | Five experimental datasets, e.g. *M. pneumoniae* (EMPIAR-10499; 65 tomograms; 198,715 subtomograms at 13.33 Å), plus simulated subtomograms of five structures (SNR 0.1–0.001, ±60°/±40° tilt). Unsupervised: there is no training/test split; ⚠️ check against which reference labels F1 was computed and whether tuning used the same data. |
| **Method** | YOPO-style 3D CNN features → PCA → GMM with K chosen by BIC → Hungarian relabelling → smoothed labels → retrain; iterate. |
| **Key equation** | $\mathrm{BIC}=p\ln n-2\ln\hat L$ to choose K; label smoothing and a hinge loss to train on pseudo-labels; DDBI as a stopping/quality criterion. |
| **Baselines** | Template matching; a convolutional autoencoder + clustering. |
| **Metrics** | Simulated: accuracy and homogeneity; experimental: F1 of ribosome detection, resolution of averages, recognisability of complexes. |
| **Reported results** | Simulated: 99.70 % vs 83.95 % (template matching) accuracy at SNR 0.1, ±60°; *M. pneumoniae*: ribosome F1 0.893 vs 0.766, ribosome average 14.17 Å; TRiC, capped proteasome, phycobilisome arrays and membranes found. |
| **Failure case** | ⚠️ Identify one from the paper: which structures or SNRs were *not* separated, and why (small size? low abundance? wedge?). |
| **Falsification test** | If clusters reflected orientation or wedge rather than structure, cluster membership would correlate with particle orientation and averages would show wedge-shaped artefacts. Test: compare orientation distributions across clusters; check averages against independent identities (e.g. template matching or labelling). |

**Blank worksheet** — copy it for each paper in [`PAPER_ATLAS.md`](../PAPER_ATLAS.md):

| Question | Paper: … |
|---|---|
| Biological question | |
| Data & split unit | |
| Method | |
| Key equation | |
| Baselines | |
| Metrics | |
| Reported results | |
| Failure case | |
| Falsification test | |

---
## 13 · From toys to real data — and doing it responsibly

**Where the real tools are.**

| Need | Where |
|---|---|
| The lab's toolkit: simulation, DoG picking, template matching, alignment, averaging, DISCA, Gum-Net, segmentation | **AITom**, [github.com/xulabs/aitom](https://github.com/xulabs/aitom) — start with `doc/tutorials/`: 004 (whole-tomogram simulation), 005 (subtomogram alignment), 008 (particle picking), 020 (Gum-Net), 021 (simple iterative average and classify), 024 (subtomogram classification) |
| Parallel subtomogram analysis at scale | **TomoMiner**, [github.com/alberlab/tomominer](https://github.com/alberlab/tomominer) (also inside AITom as `aitom/tomominer`) |
| Raw tilt series and tomograms | **EMPIAR**, [ebi.ac.uk/empiar](https://www.ebi.ac.uk/empiar/) (e.g. EMPIAR-10499 *M. pneumoniae*, EMPIAR-10988 *S. pombe*) · **CZ cryoET Data Portal**, [cryoetdataportal.czscience.com](https://cryoetdataportal.czscience.com/) |
| Maps to compare with | **EMDB**, [ebi.ac.uk/emdb](https://www.ebi.ac.uk/emdb/) (e.g. EMD-45105/45106, the fungal membranes of Module 33) |
| Atomic models | **PDB**, [rcsb.org](https://www.rcsb.org/) — the models our templates are painted from (downloaded at run time) |
| Deeper tutorials in this repository | [`TomogramDiagnosisBuild`](../../foundation/TomogramDiagnosisBuild/) · [`SubtomogramAveraging`](../../foundation/SubtomogramAveraging/) · [`SubtomogramClassification`](../../foundation/SubtomogramClassification/) · [`TomogramSegmentation`](../../foundation/TomogramSegmentation/) · [`FewShotParticleDetection`](../../foundation/FewShotParticleDetection/) |

**A path from this notebook to a first real analysis.** (1) Reproduce this pipeline with AITom's functions on a
simulated tomogram; (2) download one EMPIAR-10499 tomogram and pick with DoG; (3) classify a few thousand
subtomograms; (4) compare your ribosome average with the published one; (5) only then try something new.

**Responsible research — a checklist.**

* **Evaluate last, on data you did not tune on.** Thresholds, K and margins chosen while looking at the test data are
  not results (data leakage).
* **Pick the right unit of replication** (Module 33) — tomograms or cells, not particles, when comparing conditions.
* **Keep half-maps independent** and report the FSC criterion and mask (Module 10); beware reference bias and
  "Einstein from noise" (Module 14).
* **Match the claim to the evidence.** A 35 Å average that "looks like" a complex is a hypothesis, not an identification.
* **Report negative and failed results** (like our honest boxes in Modules 21–31): they save other people's time.
* **Share data, code and parameters** (EMPIAR/EMDB depositions, code repositories, seeds), so others can check you.
* **Credit precisely.** Say who did what; do not attribute steps to software or people when the paper does not
  (Module 33's fungal-membrane box).

---
### 📄 Paper connection — AITom (Zeng & Xu 2019; Zhan, Zeng, Uddin & Xu, J. Struct. Biol. 2025)

**Zeng X, Xu M.** *AITom: Open-source AI platform for cryo-electron tomography data analysis.* arXiv:1911.03044 (2019).
**Zhan X, Zeng X, Uddin MR, Xu M.** *AITom: AI-guided cryo-electron tomography image analyses toolkit.* J Struct Biol
217(2):108207 (2025). [doi:10.1016/j.jsb.2025.108207](https://doi.org/10.1016/j.jsb.2025.108207).
- **What it is:** the lab's open-source toolkit, grown from the Alber lab's TomoMiner library: simulation, pre-processing,
  picking (template matching, DoG, Faster R-CNN, saliency), classification (supervised, few-label, unsupervised incl. DISCA
  and Harmony), segmentation (incl. CryoSAM), interpretation (Respond-CAM), alignment and averaging (fast constrained
  alignment, FAML, Gum-Net, Jim-Net).
- **How this capstone maps onto it:** Section 1 ↔ `simulation/`; 2 ↔ `pick/dog`; 4 ↔ `classify/deep/unsupervised/disca`;
  5–6 ↔ `align/fast`, `average/align`, `classify/align`; 7 ↔ the FSC/SSNR statistics in `tomominer`.

### 📄 Paper connection — TomoMiner (Frazier, Xu & Alber, Structure 2017)

**Frazier Z, Xu M, Alber F.** *TomoMiner and TomoMinerCloud: A software platform for large-scale subtomogram structural
analysis.* Structure 25(6):951–961.e2 (2017). [doi:10.1016/j.str.2017.04.016](https://doi.org/10.1016/j.str.2017.04.016).
A C++/Python platform that runs alignment, classification, averaging and pattern mining over tens of thousands of
subtomograms with a master–worker task queue (Module 13). Our `align` loop over particles is exactly the kind of work it
distributes. ⚠️ We read the code, not the paper's text; TomoMinerCloud's details are not verified.

### 📄 Paper connection — the pipeline in review (Kim, Uddin, Xu & Chang, J. Mol. Biol. 2023)

**Kim HH-S, Uddin MR, Xu M, Chang Y-W.** *Computational methods toward unbiased pattern mining and structure
determination in cryo-electron tomography data.* J Mol Biol 435(9):168068 (2023).
[doi:10.1016/j.jmb.2023.168068](https://doi.org/10.1016/j.jmb.2023.168068).
A review organised by the same stages as this capstone — segmentation, template matching and template-free picking,
structure determination, and particle classification (supervised and unsupervised, including DISCA, multi-pattern
pursuit, Jim-Net and Harmony) — and by the open challenges: few annotations, simulation, domain shift, accessibility,
efficiency and benchmarks. It is an excellent map for deciding what to read after this course.

## ✅ Checkpoint

1. Why does the capstone keep the composition sealed until Section 11?
   <details><summary>Answer</summary>So that no choice (thresholds, K, margins) is tuned on the answer; evaluation on data you tuned on overestimates performance (data leakage).</details>
2. Why are rotation-invariant features used before alignment, and voxel comparisons after it?
   <details><summary>Answer</summary>Before alignment orientations are unknown, so only orientation-independent descriptions can be compared; after alignment the particles share a frame and full voxel-level detail becomes usable.</details>
3. Why impute missing-wedge coefficients before PCA?
   <details><summary>Answer</summary>Each aligned particle lacks a different wedge; without imputation PCA would separate particles by their wedge orientation rather than by structure.</details>
4. What makes the two half-maps "gold standard"?
   <details><summary>Answer</summary>They are refined independently from a low-resolution common start, so noise cannot be aligned consistently in both; their agreement then reflects real signal.</details>
5. How did we decide that a class was "UNKNOWN"?
   <details><summary>Answer</summary>Its best library correlation fell well below the correlation expected from its own half-map reproducibility, √(2r½/(1+r½)).</details>
6. Name two items of the audit worksheet that protect against overclaiming.
   <details><summary>Answer</summary>For example: the failure case (where the method does not work) and the falsification test (what result would show the claim is wrong); also the split unit (leakage).</details>

## 🧠 Questions & discussion

1. Our library was chosen "for a bacterium-like cell". How would a different library change the identifications and the
   discovery? What is the risk of a library that is too large?
2. The DoG elbow decided how many particles to keep. How would too few or too many picks propagate through the pipeline?
3. Our gold-standard halves started from a common 80 Å map. Why not start them from two completely different maps?
4. The segregation test found (or did not find) spatial structure. What does a non-significant result allow you to
   conclude about the biology?
5. Which step of this pipeline would you replace first with a method from the lab's recent papers, and why?

## 🚀 Bonus — capstone projects (choose one or two)

1. **Deep features.** Replace the invariant features of Section 3 with DISCA-style YOPO features (Module 22) and compare
   the ARI and the number of classes found.
2. **Deep alignment.** Replace the 400-rotation search with a Gum-Net-lite (Module 21) and compare speed and averages.
3. **Harder sample.** Add a membrane (`membrane=True`) and a second guest, or reduce the SNR; make the pipeline robust.
4. **Sparse labels.** Click 10 ribosomes and train a PU picker (Module 31); compare with DoG.
5. **Biology.** Put polysomes into the cell (Module 33) and test whether your pipeline can detect them.
6. **A fair spatial null.** Re-pack the identified composition at random 20 times with `make_crowded_cell` and redo
   Section 9 against those packings. Do the "segregation" signals disappear?
7. **Real data.** Run steps 2–7 on an EMPIAR-10499 tomogram with AITom and compare with DISCA's published ribosome.

## 📝 Lab note

```text
Module 34 — date:
Input: one sealed in silico tomogram (64 × 224 × 224 voxels of 10 Å, ±60°, defocus 5 µm, SNR 0.3)
Output: classes, averages, resolutions, identities (incl. unknowns), concentrations, report table, evaluation
Method: DoG + elbow; invariant features + PCA + GMM/BIC; wedge-aware alignment; imputation + PCA + BIC; gold-standard FSC
My abstract (written before unsealing):
What the evaluation changed in my conclusions:
The weakest step of the pipeline and how I would fix it:
Audit worksheet completed for paper:
```

## 📚 References

1. Zeng X, Xu M. AITom: Open-source AI platform for cryo-electron tomography data analysis. arXiv:1911.03044 (2019).
2. Zhan X, Zeng X, Uddin MR, Xu M. AITom: AI-guided cryo-electron tomography image analyses toolkit. *J Struct Biol* 217:108207 (2025). doi:10.1016/j.jsb.2025.108207
3. Frazier Z, Xu M, Alber F. TomoMiner and TomoMinerCloud: A software platform for large-scale subtomogram structural analysis. *Structure* 25:951–961 (2017). doi:10.1016/j.str.2017.04.016
4. Kim HH-S, Uddin MR, Xu M, Chang Y-W. Computational methods toward unbiased pattern mining and structure determination in cryo-electron tomography data. *J Mol Biol* 435:168068 (2023). doi:10.1016/j.jmb.2023.168068
5. Zeng X, Kahng A, Xue L, Mahamid J, Chang Y-W, Xu M. High-throughput cryo-ET structural pattern mining by unsupervised deep iterative subtomogram clustering. *PNAS* 120:e2213149120 (2023). doi:10.1073/pnas.2213149120
6. Pei L, Xu M, Frazier Z, Alber F. Simulating cryo electron tomograms of crowded cell cytoplasm for assessment of automated particle picking. *BMC Bioinformatics* 17:405 (2016).
7. Xu M, Beck M, Alber F. High-throughput subtomogram alignment and classification by Fourier space constrained fast volumetric matching. *J Struct Biol* 178:152–164 (2012).
8. Rosenthal PB, Henderson R. Optimal determination of particle orientation, absolute hand, and contrast loss in single-particle electron cryomicroscopy. *J Mol Biol* 333:721–745 (2003).

## ➡️ Next

🎓 **You have finished the course.** Go back to the [`README.md`](../README.md) for the reading guide and the paper atlas,
pick a paper you have not yet audited, and fill in the worksheet. Terms: [`GLOSSARY.md`](../GLOSSARY.md).